In [ ]:
!pip install scikit-learn pandas numpy scipy transformers torch datasets rapidfuzz vaderSentiment

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 122.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 126.0/126.0 kB 17.5 MB/s eta 0:00:00


In [ ]:
import os
os.makedirs('/content/drive/MyDrive/cineiq_models')


In [ ]:
import torch

In [ ]:
if torch.cuda.is_available():
  gpu=torch.cuda.get_device_name(0)
  mem=torch.cuda.get_device_properties(0).total_memory/1e9
  print(f'{gpu}({mem:.1f}gb vram)')
else:
  print("no gpu")

Tesla T4(15.6gb vram)


In [ ]:
!pip install cupy-cuda12x -q

In [ ]:
import cupy as cp
print(cp.__version__ )

14.0.1


In [2]:
import pandas as pd
import numpy as np
from scipy.sparse.linalg import svds
from scipy.sparse import csr_matrix
import pickle

# load FULL ratings — no sampling
ratings = pd.read_csv('/content/drive/MyDrive/raw/ratings.csv')
print(len(ratings))
user_ids  = sorted(ratings['userId'].unique())
movie_ids = sorted(ratings['movieId'].unique())
user_idx  = {u: i for i, u in enumerate(user_ids)}
movie_idx = {m: i for i, m in enumerate(movie_ids)}
n_users, n_movies = len(user_ids), len(movie_ids)

user_mean_series = ratings.groupby('userId')['rating'].mean()
user_means = np.array([user_mean_series[u] for u in user_ids], dtype=np.float32)

rows = ratings['userId'].map(user_idx).values.astype(np.int32)
cols = ratings['movieId'].map(movie_idx).values.astype(np.int32)
vals = ratings['rating'].values.astype(np.float32) - user_means[rows]

print(f'✓ {n_users:,} users  {n_movies:,} movies  |  mean(vals)={vals.mean():.4f}')

26024289
✓ 270,896 users  45,115 movies  |  mean(vals)=0.0000


In [ ]:
# Cell 6 — Build sparse matrix on GPU
import cupy as cp
from cupyx.scipy.sparse import csr_matrix as gpu_csr

sparse_gpu = gpu_csr(
    (cp.array(vals), (cp.array(rows), cp.array(cols))),
    shape=(n_users, n_movies),
    dtype=cp.float32
)
print(f'✓ Sparse matrix on GPU: {sparse_gpu.shape}  nnz={sparse_gpu.nnz:,}')

✓ Sparse matrix on GPU: (270896, 45115)  nnz=26,024,289


In [ ]:
# Cell 7 — Run SVD on GPU (~2-4 min on T4)
from cupyx.scipy.sparse.linalg import svds as gpu_svds
import time

k = min(100, min(n_users, n_movies) - 1)
print(f'Running svds(k={k})...')

t0 = time.time()
U_gpu, sigma_gpu, Vt_gpu = gpu_svds(sparse_gpu, k=k)
print(f'✓ Done in {time.time()-t0:.1f}s')

# svds returns ascending order — reverse
order     = cp.argsort(sigma_gpu)[::-1]
sigma_gpu = sigma_gpu[order]
U_gpu     = U_gpu[:, order]
Vt_gpu    = Vt_gpu[order, :]

print(f'Top 5 singular values: {cp.asnumpy(sigma_gpu[:5]).round(1)}')

Running svds(k=100)...
✓ Done in 5.2s
Top 5 singular values: [1016.3  529.5  458.5  424.1  388.5]


In [ ]:
# Cell 8 — Save to Drive
import pickle, os

svd_data = {
    'U':          cp.asnumpy(U_gpu).astype(np.float32),
    'sigma':      cp.asnumpy(sigma_gpu).astype(np.float32),
    'Vt':         cp.asnumpy(Vt_gpu).astype(np.float32),
    'user_ids':   user_ids,
    'movie_ids':  movie_ids,
    'user_idx':   user_idx,
    'movie_idx':  movie_idx,
    'user_means': user_means,
}

with open('/content/drive/MyDrive/cineiq_models/svd_model.pkl', 'wb') as f:
    pickle.dump(svd_data, f)

#print(f'✓ Saved {os.path.getsize(SAVE_PATH)/1e6:.0f} MB to {SAVE_PATH}')

In [ ]:
import ast
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import scipy.sparse
import pickle
import pandas as pd

def extract_genres(genre_str):
    try:
        genres = ast.literal_eval(genre_str)
        return ' '.join([g['name'] for g in genres])
    except:
        return ''

def extract_keywords(kw_str):
    try:
        items = ast.literal_eval(kw_str)
        return ' '.join([k['name'].replace(' ','') for k in items])
    except:
        return ''

def extract_cast(cast_str, n=3):
    try:
        cast = ast.literal_eval(cast_str)
        return ' '.join([c['name'].replace(' ','') for c in cast[:n]])
    except:
        return ''

def extract_director(crew_str):
    try:
        crew = ast.literal_eval(crew_str)
        for c in crew:
            if c['job'] == 'Director':
                return c['name'].replace(' ','')
        return ''
    except:
        return ''

# load TMDB
tmdb = pd.read_csv('/content/drive/MyDrive/raw/movies_metadata.csv', low_memory=False)
tmdb = tmdb[pd.to_numeric(tmdb['id'], errors='coerce').notna()]
tmdb['id'] = tmdb['id'].astype(int)
tmdb['genre_names'] = tmdb['genres'].apply(extract_genres)
tmdb = tmdb[['id','title','genre_names','overview',
             'release_date','vote_average',
             'vote_count','original_language']].copy()

# merge keywords
kw = pd.read_csv('/content/drive/MyDrive/raw/keywords.csv')
kw['keyword_names'] = kw['keywords'].apply(extract_keywords)
tmdb = tmdb.merge(kw[['id','keyword_names']], on='id', how='left')
tmdb['keyword_names'] = tmdb['keyword_names'].fillna('')

# merge credits
cr = pd.read_csv('/content/drive/MyDrive/raw/credits.csv')
cr['cast_names']    = cr['cast'].apply(extract_cast)
cr['director_name'] = cr['crew'].apply(extract_director)
tmdb = tmdb.merge(
    cr[['id','cast_names','director_name']],
    on='id', how='left'
)
tmdb['cast_names']    = tmdb['cast_names'].fillna('')
tmdb['director_name'] = tmdb['director_name'].fillna('')
tmdb['overview']      = tmdb['overview'].fillna('')
tmdb['original_language']=tmdb['original_language'].fillna('')
tmdb['vote_count']    = pd.to_numeric(
    tmdb['vote_count'], errors='coerce'
).fillna(0)

# build soup
tmdb['soup'] = (
    tmdb['original_language']+' '+
    tmdb['original_language']+' '+
    tmdb['original_language']+' '+
    tmdb['genre_names']   + ' ' +
    tmdb['genre_names']   + ' ' +
    tmdb['genre_names']   + ' ' +
    tmdb['genre_names']   + ' ' +
    tmdb['genre_names']   + ' ' +
    tmdb['genre_names']   + ' ' +
    tmdb['overview']   + ' ' +
    tmdb['overview']   + ' ' +
    tmdb['keyword_names'] + ' ' +
    tmdb['keyword_names'] + ' ' +
    tmdb['keyword_names'] + ' ' +
    tmdb['keyword_names'] + ' ' +
    tmdb['keyword_names'] + ' ' +
    tmdb['director_name'] + ' ' +
    tmdb['director_name'] + ' ' +
    tmdb['director_name'] + ' ' +
    tmdb['cast_names']    + ' ' +
    tmdb['cast_names']    + ' ' +
    tmdb['cast_names']    + ' ' +
    tmdb['cast_names']
)

tmdb['soup_len'] = tmdb['soup'].str.split().str.len()
tmdb = tmdb[tmdb['soup_len'] >= 20].reset_index(drop=True)
print(f"Movies after filtering: {len(tmdb)}")

# build TF-IDF
tfidf = TfidfVectorizer(
    stop_words='english',
    max_features=20000,
    ngram_range=(1, 2)
)
tfidf_matrix = tfidf.fit_transform(tmdb['soup'])
print(f"✓ TF-IDF matrix: {tfidf_matrix.shape}")

# save everything
tmdb.to_pickle(
    '/content/drive/MyDrive/cineiq_models/tmdb_clean.pkl'
)
scipy.sparse.save_npz(
    '/content/drive/MyDrive/cineiq_models/tfidf_matrix.npz',
    tfidf_matrix
)
with open('/content/drive/MyDrive/cineiq_models/tfidf_vectorizer.pkl','wb') as f:
    pickle.dump(tfidf, f)

print("✓ TF-IDF saved to Drive")

Movies after filtering: 46291
✓ TF-IDF matrix: (46291, 20000)
✓ TF-IDF saved to Drive


In [ ]:
import torch
import pandas as pd
from torch.utils.data import Dataset, DataLoader
from transformers import (DistilBertTokenizerFast,
                          DistilBertForSequenceClassification)
from torch.optim import AdamW
import re

# check GPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")
# should print "Device: cuda" on Colab

def clean_review(text):
    text = re.sub(r'<[^>]+>', ' ', text)
    return re.sub(r'\s+', ' ', text).strip()

imdb = pd.read_csv('/content/drive/MyDrive/raw/IMDB Dataset.csv')
imdb['review'] = imdb['review'].apply(clean_review)
imdb['label']  = (imdb['sentiment'] == 'positive').astype(int)
print(f"IMDB: {imdb.shape}")

class IMDBDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=256):
        self.texts     = texts
        self.labels    = labels
        self.tokenizer = tokenizer
        self.max_len   = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.texts[idx],
            truncation=True,
            padding='max_length',
            max_length=self.max_len,
            return_tensors='pt'
        )
        return {
            'input_ids':      enc['input_ids'].squeeze(),
            'attention_mask': enc['attention_mask'].squeeze(),
            'labels':          torch.tensor(
                self.labels[idx], dtype=torch.long
            )
        }

tokenizer = DistilBertTokenizerFast.from_pretrained(
    'distilbert-base-uncased'
)
model = DistilBertForSequenceClassification.from_pretrained(
    'distilbert-base-uncased', num_labels=2
)
model.to(device)

# use FULL 50K dataset — split 80/20
df_shuffled = imdb.sample(frac=1, random_state=42).reset_index(drop=True)
train_df    = df_shuffled[:40000]
val_df      = df_shuffled[40000:]

train_ds = IMDBDataset(
    train_df['review'].tolist(),
    train_df['label'].tolist(),
    tokenizer
)
val_ds = IMDBDataset(
    val_df['review'].tolist(),
    val_df['label'].tolist(),
    tokenizer
)

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)
val_loader   = DataLoader(val_ds,   batch_size=32)

optimizer = AdamW(model.parameters(), lr=2e-5)

# train 3 epochs on full data
for epoch in range(3):
    model.train()
    total_loss = 0
    correct    = 0
    total      = 0

    for batch_idx, batch in enumerate(train_loader):
        input_ids      = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels         = batch['labels'].to(device)

        outputs = model(input_ids=input_ids,
                       attention_mask=attention_mask,
                       labels=labels)
        loss    = outputs.loss
        logits  = outputs.logits

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        preds       = torch.argmax(logits, dim=1)
        correct    += (preds == labels).sum().item()
        total      += labels.size(0)

        if (batch_idx + 1) % 100 == 0:
            print(f"  Epoch {epoch+1} | Batch {batch_idx+1} | "
                  f"Loss: {total_loss/(batch_idx+1):.4f} | "
                  f"Acc: {correct/total:.1%}")

    # validate
    model.eval()
    val_correct = 0
    val_total   = 0
    with torch.no_grad():
        for batch in val_loader:
            input_ids      = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels         = batch['labels'].to(device)
            outputs        = model(input_ids=input_ids,
                                  attention_mask=attention_mask)
            preds          = torch.argmax(outputs.logits, dim=1)
            val_correct   += (preds == labels).sum().item()
            val_total     += labels.size(0)

    print(f"\n✓ Epoch {epoch+1} Val Accuracy: "
          f"{val_correct/val_total:.1%}\n")

# save to Drive
save_path = '/content/drive/MyDrive/cineiq_models/distilbert_sentiment'
model.save_pretrained(save_path)
tokenizer.save_pretrained(save_path)
print(f"✓ DistilBERT saved to {save_path}")

Device: cuda
IMDB: (50000, 3)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


  Epoch 1 | Batch 100 | Loss: 0.4896 | Acc: 75.6%
  Epoch 1 | Batch 200 | Loss: 0.3924 | Acc: 81.5%
  Epoch 1 | Batch 300 | Loss: 0.3549 | Acc: 83.6%
  Epoch 1 | Batch 400 | Loss: 0.3336 | Acc: 84.9%
  Epoch 1 | Batch 500 | Loss: 0.3169 | Acc: 85.8%
  Epoch 1 | Batch 600 | Loss: 0.3071 | Acc: 86.4%
  Epoch 1 | Batch 700 | Loss: 0.2970 | Acc: 86.9%
  Epoch 1 | Batch 800 | Loss: 0.2884 | Acc: 87.4%
  Epoch 1 | Batch 900 | Loss: 0.2826 | Acc: 87.8%
  Epoch 1 | Batch 1000 | Loss: 0.2794 | Acc: 87.9%
  Epoch 1 | Batch 1100 | Loss: 0.2741 | Acc: 88.2%
  Epoch 1 | Batch 1200 | Loss: 0.2706 | Acc: 88.4%

✓ Epoch 1 Val Accuracy: 91.5%

  Epoch 2 | Batch 100 | Loss: 0.1622 | Acc: 94.1%
  Epoch 2 | Batch 200 | Loss: 0.1622 | Acc: 93.9%
  Epoch 2 | Batch 300 | Loss: 0.1598 | Acc: 94.2%
  Epoch 2 | Batch 400 | Loss: 0.1618 | Acc: 94.1%
  Epoch 2 | Batch 500 | Loss: 0.1605 | Acc: 94.1%
  Epoch 2 | Batch 600 | Loss: 0.1629 | Acc: 94.0%
  Epoch 2 | Batch 700 | Loss: 0.1615 | Acc: 94.0%
  Epoch 2 | Bat

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

✓ DistilBERT saved to /content/drive/MyDrive/cineiq_models/distilbert_sentiment


In [1]:
import pandas as pd
import numpy as np
from sklearn.cluster import MiniBatchKMeans
from sklearn.preprocessing import normalize
import pickle

print("Loading ratings...")
ratings = pd.read_csv('/content/drive/MyDrive/raw/ratings.csv')
print(f"Full ratings: {ratings.shape}")

# keep top 3000 movies and top 50000 most active users
# more users = better clusters
top_movies = ratings['movieId'].value_counts().head(3000).index
top_users  = ratings['userId'].value_counts().head(50000).index

filtered = ratings[
    ratings['movieId'].isin(top_movies) &
    ratings['userId'].isin(top_users)
]

print(f"Filtered: {filtered['userId'].nunique()} users "
      f"x {filtered['movieId'].nunique()} movies")

# build user-movie matrix
print("Building user-movie matrix...")
user_movie = filtered.pivot_table(
    index='userId', columns='movieId', values='rating'
).fillna(0)

user_ids  = list(user_movie.index)
mat       = user_movie.values.astype(np.float32)

# normalise rows so clustering is based on taste
# not on how many movies someone rated
print("Normalising...")
mat_norm = normalize(mat, norm='l2')

# cluster users into 50 taste groups
# MiniBatchKMeans is fast enough for 20K users
print("Clustering users into 100 groups...")
n_clusters = 100
kmeans = MiniBatchKMeans(
    n_clusters  = n_clusters,
    random_state= 42,
    batch_size  = 2000,
    n_init      = 10
)
cluster_labels = kmeans.fit_predict(mat_norm)
print(f"✓ Clustering done")

# print cluster sizes
unique, counts = np.unique(cluster_labels, return_counts=True)
print(f"  Cluster sizes: min={counts.min()} "
      f"max={counts.max()} mean={counts.mean():.0f}")

# build cluster lookup
# user_id → cluster_id
user_cluster = {
    user_ids[i]: int(cluster_labels[i])
    for i in range(len(user_ids))
}

# cluster → list of user_ids in that cluster
cluster_users = {}
for user_id, cluster_id in user_cluster.items():
    if cluster_id not in cluster_users:
        cluster_users[cluster_id] = []
    cluster_users[cluster_id].append(user_id)

cluster_data = {
    'user_cluster':   user_cluster,
    'cluster_users':  cluster_users,
    'user_ids':       user_ids,
    'movie_ids':      list(user_movie.columns),
    'cluster_labels': cluster_labels,
    'n_clusters':     n_clusters,
    'kmeans':         kmeans
}

with open('/content/drive/MyDrive/cineiq_models/cluster_data.pkl', 'wb') as f:
    pickle.dump(cluster_data, f)

print("✓ Cluster data saved to Drive")
print(f"  Total users clustered: {len(user_cluster)}")
print(f"  Total clusters: {n_clusters}")

Loading ratings...
Full ratings: (26024289, 4)
26024289


In [ ]:
!pip install transformers torch vaderSentiment pandas -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 126.0/126.0 kB 5.4 MB/s eta 0:00:00


In [ ]:
import torch

In [ ]:
print(torch.cuda.get_device_name(0))

Tesla T4


In [ ]:
from google.colab import files
uploaded=files.upload()

Saving IMDB Dataset.csv to IMDB Dataset.csv


In [ ]:
import pandas as pd
import numpy as np
import re
from pathlib import Path
import math

# ── VADER ─────────────────────────────────────────────────────
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

# ── paths ──────────────────────────────────────────────────────
DATA_RAW  = Path("/content/drive/MyDrive")
MODEL_DIR = Path("models/distilbert_sentiment")

# The split index is fixed so train and eval NEVER overlap.
# 5000 train  ·  1000 val  ·  remainder (~43 000) = eval pool
_TRAIN_END = 5000
_VAL_END   = 6000   # indices [5000, 6000) are validation
# evaluate.py samples from indices [6000, …) — never seen during training


# ══════════════════════════════════════════════════════════════
# UTILITIES
# ══════════════════════════════════════════════════════════════

def clean_review(text):
    text = re.sub(r'<[^>]+>', ' ', text)
    text = re.sub(r'\s+', ' ', text)
    return text.strip()


def load_imdb():
    path = "IMDB Dataset.csv"
    df   = pd.read_csv(path)
    df['review'] = df['review'].apply(clean_review)
    df['label']  = (df['sentiment'] == 'positive').astype(int)
    print(f"✓ IMDB loaded and cleaned: {df.shape}")
    return df


def get_imdb_splits(df):
    """
    Return (train_df, val_df, eval_df) with guaranteed zero overlap.

    Uses a fixed seed shuffle so splits are reproducible across
    all scripts, but the seed is applied ONCE here — not scattered
    across load_imdb / train / evaluate calls.

    train : rows [0,      _TRAIN_END)  → fine-tuning only
    val   : rows [_TRAIN_END, _VAL_END) → used during training loop
    eval  : rows [_VAL_END,  end)       → evaluate.py samples from here
    """
    shuffled  = df.sample(frac=1, random_state=0).reset_index(drop=True)
    train_df  = shuffled.iloc[:_TRAIN_END].copy()
    val_df    = shuffled.iloc[_TRAIN_END:_VAL_END].copy()
    eval_df   = shuffled.iloc[_VAL_END:].copy()
    print(f"✓ IMDB split → train={len(train_df):,}  "
          f"val={len(val_df):,}  eval={len(eval_df):,}")
    return train_df, val_df, eval_df


# ══════════════════════════════════════════════════════════════
# 1. VADER
# ══════════════════════════════════════════════════════════════

def build_vader():
    analyzer = SentimentIntensityAnalyzer()
    print("✓ VADER analyzer ready")
    return analyzer


def vader_score(text, analyzer):
    return analyzer.polarity_scores(text)['compound']


def vader_score_list(reviews, analyzer):
    if not reviews:
        return 0.0
    return float(np.mean([vader_score(r, analyzer) for r in reviews]))


def evaluate_vader(df, analyzer, sample=500):
    sample_df = df.sample(n=min(sample, len(df)), random_state=1)
    correct   = sum(
        1 for _, row in sample_df.iterrows()
        if (1 if vader_score(row['review'], analyzer) > 0 else 0)
           == row['label']
    )
    accuracy = correct / len(sample_df)
    print(f"✓ VADER accuracy on {len(sample_df)} IMDB reviews: {accuracy:.1%}")
    return accuracy


# ══════════════════════════════════════════════════════════════
# 2. RE-RANKER
# ══════════════════════════════════════════════════════════════

def rerank_with_sentiment(hybrid_recs, analyzer,
                           tmdb_df=None,
                           distilbert_model=None,
                           distilbert_tokenizer=None,
                           hybrid_weight=0.70,
                           tmdb_weight=0.15,
                           distilbert_weight=0.15):
    assert abs(hybrid_weight + tmdb_weight + distilbert_weight - 1.0) < 1e-6, \
        "weights must sum to 1.0"

    print(f"\nRe-ranking {len(hybrid_recs)} recommendations "
          f"(hybrid={hybrid_weight}, tmdb={tmdb_weight}, "
          f"distilbert={distilbert_weight})...")

    results = []
    for rec in hybrid_recs:
        title        = rec['title']
        tmdb_score   = 0.5
        overview_text = ''

        if tmdb_df is not None:
            row = tmdb_df[tmdb_df['title'].str.lower() == title.lower()]
            if len(row) == 0:
                clean = re.sub(r'\s*\(\d{4}\)\s*$', '',
                               title).strip().lower()
                row = tmdb_df[tmdb_df['title'].str.lower() == clean]

            if len(row) > 0:
                r          = row.iloc[0]
                vote_avg   = float(r.get('vote_average', 5) or 5)
                vote_count = float(r.get('vote_count',   0) or 0)
                quality    = vote_avg / 10.0
                confidence = min(math.log10(vote_count + 1) / 5.0, 1.0)
                tmdb_score = round(quality * confidence, 4)
                overview_text = str(r.get('overview', '') or '')

        db_score = 0.5
        if overview_text:
            if distilbert_model is not None and distilbert_tokenizer is not None:
                try:
                    db_score = distilbert_score(
                        overview_text, distilbert_model, distilbert_tokenizer
                    )
                except Exception:
                    db_score = 0.5
            else:
                compound = analyzer.polarity_scores(overview_text)['compound']
                db_score = round((compound + 1) / 2, 4)

        final_score = round(
            hybrid_weight     * rec['hybrid_score'] +
            tmdb_weight       * tmdb_score          +
            distilbert_weight * db_score,
            4
        )

        results.append({
            'title':            rec['title'],
            'hybrid_score':     rec['hybrid_score'],
            'tmdb_score':       tmdb_score,
            'distilbert_score': round(db_score, 4),
            'sentiment_score':  round(db_score, 4),
            'final_score':      final_score
        })

    results.sort(key=lambda x: x['final_score'], reverse=True)
    return results


# ══════════════════════════════════════════════════════════════
# 3. DISTILBERT
# ══════════════════════════════════════════════════════════════
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import (DistilBertTokenizerFast,
                          DistilBertForSequenceClassification)
from torch.optim import AdamW


class IMDBDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=256):
        self.texts     = texts
        self.labels    = labels
        self.tokenizer = tokenizer
        self.max_len   = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            self.texts[idx],
            truncation=True,
            padding='max_length',
            max_length=self.max_len,
            return_tensors='pt'
        )
        return {
            'input_ids':      encoding['input_ids'].squeeze(),
            'attention_mask': encoding['attention_mask'].squeeze(),
            'label':          torch.tensor(self.labels[idx], dtype=torch.long)
        }


def train_distilbert(df, epochs=2, batch_size=16, max_length=256):
    """
    Fine-tune DistilBERT using the fixed train/val split from
    get_imdb_splits().  The eval split is never touched here.
    """
    train_df, val_df, _ = get_imdb_splits(df)   # eval split ignored

    print("\nFine-tuning DistilBERT...")
    print(f"  Train: {len(train_df):,} | Val: {len(val_df):,} | "
          f"Epochs: {epochs} | Batch: {batch_size}")

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"  Device: {device}")

    tokenizer = DistilBertTokenizerFast.from_pretrained(
        'distilbert-base-uncased'
    )
    model = DistilBertForSequenceClassification.from_pretrained(
        'distilbert-base-uncased', num_labels=2
    )
    model.to(device)

    train_dataset = IMDBDataset(
        train_df['review'].tolist(),
        train_df['label'].tolist(),
        tokenizer, max_length
    )
    val_dataset = IMDBDataset(
        val_df['review'].tolist(),
        val_df['label'].tolist(),
        tokenizer, max_length
    )

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    val_loader   = DataLoader(val_dataset,   batch_size=batch_size)

    optimizer = AdamW(model.parameters(), lr=2e-5)

    for epoch in range(epochs):
        model.train()
        total_loss = 0
        correct    = 0
        total      = 0

        for batch_idx, batch in enumerate(train_loader):
            input_ids      = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels         = batch['label'].to(device)

            outputs = model(input_ids=input_ids,
                            attention_mask=attention_mask,
                            labels=labels)
            loss   = outputs.loss
            logits = outputs.logits

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            total_loss += loss.item()
            preds       = torch.argmax(logits, dim=1)
            correct    += (preds == labels).sum().item()
            total      += labels.size(0)

            if (batch_idx + 1) % 50 == 0:
                print(f"    Epoch {epoch+1} | "
                      f"Batch {batch_idx+1}/{len(train_loader)} | "
                      f"Loss: {total_loss/(batch_idx+1):.4f} | "
                      f"Train Acc: {correct/total:.1%}")

        model.eval()
        val_correct = 0
        val_total   = 0

        with torch.no_grad():
            for batch in val_loader:
                input_ids      = batch['input_ids'].to(device)
                attention_mask = batch['attention_mask'].to(device)
                labels         = batch['label'].to(device)

                outputs = model(input_ids=input_ids,
                                attention_mask=attention_mask)
                preds   = torch.argmax(outputs.logits, dim=1)
                val_correct += (preds == labels).sum().item()
                val_total   += labels.size(0)

        val_acc = val_correct / val_total
        print(f"\n  ✓ Epoch {epoch+1} complete | Val Accuracy: {val_acc:.1%}\n")

    MODEL_DIR.mkdir(parents=True, exist_ok=True)
    model.save_pretrained(MODEL_DIR)
    tokenizer.save_pretrained(MODEL_DIR)
    print(f"✓ DistilBERT saved to {MODEL_DIR}")
    return model, tokenizer


def load_distilbert():
    device    = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    tokenizer = DistilBertTokenizerFast.from_pretrained(MODEL_DIR)
    model     = DistilBertForSequenceClassification.from_pretrained(MODEL_DIR)
    model.to(device)
    model.eval()
    print(f"✓ DistilBERT loaded from {MODEL_DIR}")
    return model, tokenizer


def distilbert_score(text, model, tokenizer, max_length=256):
    """Returns probability of positive sentiment (0–1)."""
    device   = next(model.parameters()).device
    encoding = tokenizer(
        text,
        truncation=True,
        padding='max_length',
        max_length=max_length,
        return_tensors='pt'
    )
    input_ids      = encoding['input_ids'].to(device)
    attention_mask = encoding['attention_mask'].to(device)

    with torch.no_grad():
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        probs   = torch.softmax(outputs.logits, dim=1)
        return float(probs[0][1])


def evaluate_distilbert(df, model, tokenizer, sample=200):
    """Test on held-out eval split only — never touches train/val rows."""
    _, _, eval_df = get_imdb_splits(df)
    sample_df = eval_df.sample(n=min(sample, len(eval_df)), random_state=2)
    correct   = 0

    for _, row in sample_df.iterrows():
        score     = distilbert_score(row['review'], model, tokenizer)
        predicted = 1 if score > 0.5 else 0
        if predicted == row['label']:
            correct += 1

    accuracy = correct / len(sample_df)
    print(f"✓ DistilBERT accuracy on {len(sample_df)} held-out reviews: {accuracy:.1%}")
    return accuracy


# ══════════════════════════════════════════════════════════════
# MAIN
# ══════════════════════════════════════════════════════════════

if __name__ == "__main__":
    print("=" * 55)
    print("PHASE 4 — Sentiment Analysis")
    print("=" * 55)

    imdb     = load_imdb()
    analyzer = build_vader()

    print("\n--- VADER on sample reviews ---")
    for i in range(3):
        review = imdb['review'].iloc[i]
        score  = vader_score(review, analyzer)
        actual = imdb['sentiment'].iloc[i]
        print(f"\n  Review snippet : {review[:80]}...")
        print(f"  VADER compound : {score:.4f}")
        print(f"  Actual label   : {actual}")
        print(f"  VADER predicted: {'positive' if score > 0 else 'negative'}")

    print("\n--- VADER accuracy ---")
    _, _, eval_df = get_imdb_splits(imdb)
    evaluate_vader(eval_df, analyzer, sample=500)

    print("\n--- DistilBERT fine-tuning ---")
    model, tokenizer = train_distilbert(imdb, epochs=2, batch_size=16)

    print("\n--- DistilBERT accuracy on held-out eval set ---")
    evaluate_distilbert(imdb, model, tokenizer, sample=300)

PHASE 4 — Sentiment Analysis
✓ IMDB loaded and cleaned: (50000, 3)
✓ VADER analyzer ready

--- VADER on sample reviews ---

  Review snippet : One of the other reviewers has mentioned that after watching just 1 Oz episode y...
  VADER compound : -0.9916
  Actual label   : positive
  VADER predicted: negative

  Review snippet : A wonderful little production. The filming technique is very unassuming- very ol...
  VADER compound : 0.9670
  Actual label   : positive
  VADER predicted: positive

  Review snippet : I thought this was a wonderful way to spend time on a too hot summer weekend, si...
  VADER compound : 0.9745
  Actual label   : positive
  VADER predicted: positive

--- VADER accuracy ---
✓ IMDB split → train=5,000  val=1,000  eval=44,000
✓ VADER accuracy on 500 IMDB reviews: 69.2%

--- DistilBERT fine-tuning ---
✓ IMDB split → train=5,000  val=1,000  eval=44,000

Fine-tuning DistilBERT...
  Train: 5,000 | Val: 1,000 | Epochs: 2 | Batch: 16
  Device: cuda


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


    Epoch 1 | Batch 50/313 | Loss: 0.5600 | Train Acc: 70.6%
    Epoch 1 | Batch 100/313 | Loss: 0.4792 | Train Acc: 77.1%
    Epoch 1 | Batch 150/313 | Loss: 0.4235 | Train Acc: 80.4%
    Epoch 1 | Batch 200/313 | Loss: 0.4123 | Train Acc: 81.5%
    Epoch 1 | Batch 250/313 | Loss: 0.3918 | Train Acc: 82.5%
    Epoch 1 | Batch 300/313 | Loss: 0.3759 | Train Acc: 83.6%

  ✓ Epoch 1 complete | Val Accuracy: 88.6%

    Epoch 2 | Batch 50/313 | Loss: 0.2271 | Train Acc: 91.6%
    Epoch 2 | Batch 100/313 | Loss: 0.2226 | Train Acc: 91.4%
    Epoch 2 | Batch 150/313 | Loss: 0.2012 | Train Acc: 92.3%
    Epoch 2 | Batch 200/313 | Loss: 0.2042 | Train Acc: 92.2%
    Epoch 2 | Batch 250/313 | Loss: 0.2016 | Train Acc: 92.4%
    Epoch 2 | Batch 300/313 | Loss: 0.1991 | Train Acc: 92.5%

  ✓ Epoch 2 complete | Val Accuracy: 90.3%



Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

✓ DistilBERT saved to models/distilbert_sentiment

--- DistilBERT accuracy on held-out eval set ---
✓ IMDB split → train=5,000  val=1,000  eval=44,000
✓ DistilBERT accuracy on 300 held-out reviews: 93.3%


In [ ]:
import shutil

In [ ]:
shutil.copytree("models/distilbert_sentiment","/content/drive/MyDrive/distilbert_sentiment")

'/content/drive/MyDrive/distilbert_sentiment'